# Hiver SDE Intern — AppleSupport AI Agent (Simple Pipeline)

**Brand:** AppleSupport
**Approach:** Gemini 3.6 Flash does the reasoning (classification + reply drafting + judging). Classic ML (TF-IDF + Logistic Regression) is kept only as a *simple baseline* to compare against, as the assignment requires.
**Retrieval:** TF-IDF + cosine similarity over historically resolved AppleSupport threads, used to ground replies.
**Golden set:** 150-250 hand-labelled examples, kept fully separate from anything the model sees during development.

This notebook is intentionally lean: no neural network training, no large LLM-labeled training set. Everything needed for the assignment's deliverables is produced by running this notebook top to bottom, in well under 15 minutes (excluding the one-time Kaggle download).

**What this notebook produces:**
1. `golden_set_to_label.csv` - template for you to hand-label 150-250 examples (one time).
2. Baselines: majority-class and TF-IDF+LogisticRegression, evaluated on the golden set.
3. Gemini-based intent classifier, reply drafter (grounded in retrieved past resolutions), and auto/escalate policy - evaluated on the same golden set.
4. An LLM-as-judge rubric for reply quality, with a small human-agreement check.
5. Failure analysis (top confusions with real examples).
6. Saved artifacts + a tiny Gradio demo you can click through live.


## 0. Setup

Run this cell once. It installs dependencies and asks you to paste your Gemini API key directly (no Colab Secrets needed).

In [ ]:
%pip -q install -U kagglehub google-genai pydantic scikit-learn pandas numpy gradio


In [ ]:
import re
import json
import getpass
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity

import joblib

SEED = 42
np.random.seed(SEED)

print("Environment ready.")


### Paste your Gemini API key here

Get a key from https://aistudio.google.com/apikey . The cell below hides your typing (`getpass`) so it isn't shown in cell output, but it **will** still be held in the notebook's execution state for this session - never commit a notebook with the key hard-coded into a plain cell, and restart the runtime (Runtime -> Restart session) before sharing this notebook after running it.

In [ ]:
GEMINI_API_KEY = getpass.getpass("Paste your Gemini API key: ")

from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Literal

client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.6-flash"

print("Gemini client ready.")


## 1. Download and load the dataset

Primary dataset per the assignment: `thoughtvector/customer-support-on-twitter` (Kaggle). We download it via `kagglehub` so no manual upload of the ~3M-row CSV is needed.

> First time running this in Colab, `kagglehub` may ask you to authenticate with a Kaggle account/API token.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")

csv_candidates = list(Path(dataset_path).rglob("twcs.csv"))
if not csv_candidates:
    raise FileNotFoundError("twcs.csv was not found in the downloaded dataset.")
DATA_FILE = str(csv_candidates[0])

USE_COLS = [
    "tweet_id", "author_id", "inbound", "created_at",
    "text", "response_tweet_id", "in_response_to_tweet_id",
]

df = pd.read_csv(DATA_FILE, usecols=USE_COLS, low_memory=False)
print("Full dataset:", df.shape)
df.head()


## 2. Pick a brand and reconstruct customer -> brand reply pairs

We use **AppleSupport**. Change `BRAND` below to try a different brand.

For every brand reply, we follow `in_response_to_tweet_id` back into the **full** dataset (not just the brand's own rows) and keep the pair only when the parent tweet is genuinely an inbound customer message. This avoids the common bug of accidentally pairing brand-to-brand tweets.

In [ ]:
BRAND = "AppleSupport"

brand_replies = df[(df["author_id"] == BRAND) & (df["inbound"] == False)].copy()
brand_replies = brand_replies.rename(columns={"tweet_id": "reply_tweet_id", "text": "brand_reply"})

parent_lookup = df[["tweet_id", "author_id", "inbound", "text"]].rename(columns={
    "tweet_id": "customer_tweet_id",
    "author_id": "customer_author_id",
    "inbound": "customer_inbound",
    "text": "customer_text",
})

pairs = brand_replies.merge(
    parent_lookup, left_on="in_response_to_tweet_id", right_on="customer_tweet_id", how="inner"
)
pairs = pairs[pairs["customer_inbound"] == True].copy()
pairs = pairs[[
    "reply_tweet_id", "customer_tweet_id", "customer_author_id", "customer_text", "brand_reply"
]].reset_index(drop=True)

print(f"Direct customer -> {BRAND} pairs:", len(pairs))
pairs.head(5)


In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = (text.replace("&amp;", "&").replace("&lt;", "<")
                .replace("&gt;", ">").replace("&quot;", '"'))
    text = re.sub(r"\s+", " ", text).strip()
    return text

pairs["clean_customer_text"] = pairs["customer_text"].apply(clean_text)
pairs["clean_brand_reply"] = pairs["brand_reply"].apply(clean_text)

pairs = pairs[pairs["clean_customer_text"].str.len() >= 8]
pairs = pairs.drop_duplicates(subset=["clean_customer_text"]).reset_index(drop=True)

print("Usable pairs after cleaning/dedup:", len(pairs))

# Keep the pipeline fast: work on a manageable, representative subsample.
# The assignment explicitly expects and encourages this.
SUBSAMPLE_SIZE = 4000
pairs_sample = pairs.sample(min(SUBSAMPLE_SIZE, len(pairs)), random_state=SEED).reset_index(drop=True)
print("Working subsample size:", len(pairs_sample))


## 3. Intent taxonomy

11 intents grounded in what actually shows up in AppleSupport conversations. Rule: label the customer's **main problem**, not a keyword mentioned in passing.

In [ ]:
INTENTS = [
    "ios_update_issue", "device_performance", "keyboard_display_bug",
    "apps_services_issue", "connectivity_issue", "account_icloud_issue",
    "hardware_issue", "photos_data_issue", "how_to_information",
    "complaint_feedback", "other",
]

INTENT_DEFINITIONS = {
    "ios_update_issue": "Problems installing, downloading, completing, or performing an iOS/software update.",
    "device_performance": "Freezing, crashing, restarting, shutting down, lagging, or slow/unresponsive device behavior.",
    "keyboard_display_bug": "Keyboard, autocorrect, text-entry, character-rendering, or display-glitch problems.",
    "apps_services_issue": "Problems with apps/services such as iTunes, Apple Music, App Store, notifications, reminders.",
    "connectivity_issue": "Wi-Fi, Bluetooth, cellular, USB, calling, or other device connection problems.",
    "account_icloud_issue": "Apple ID, iCloud, login, password, account recovery, or account-access problems.",
    "hardware_issue": "Battery, charging, speaker, screen, camera, microphone, or other physical-device problems.",
    "photos_data_issue": "Lost/deleted photos, backup/restore problems, missing data, or transfer/data-loss issues.",
    "how_to_information": "A direct how-to question or general product-information request.",
    "complaint_feedback": "Primarily dissatisfaction, frustration, or criticism without a more specific technical intent.",
    "other": "Insufficient context, acknowledgement, unrelated content, or genuinely ambiguous messages.",
}

for name in INTENTS:
    print(f"- {name}: {INTENT_DEFINITIONS[name]}")


## 4. Golden evaluation set (150-250 hand-labelled examples)

This is the ground truth you build yourself - it must stay independent of anything used to prompt or tune the model.

Run the cell below once. If `golden_set_labeled.csv` doesn't exist yet, it creates `golden_set_to_label.csv`, a sample of 200 real customer messages for you to hand-label with:
- `true_intent` - one of the `INTENTS` above
- `expected_action` - `auto` or `escalate`
- `label_notes` - optional, why you chose that label

Label it (in Excel/Sheets/a text editor), save/upload it as `golden_set_labeled.csv`, then re-run the cell - it will load your labels instead.

In [ ]:
GOLDEN_SIZE = 200
golden_labeled_path = Path("golden_set_labeled.csv")
golden_template_path = Path("golden_set_to_label.csv")

if golden_labeled_path.exists():
    golden = pd.read_csv(golden_labeled_path)
    print("Loaded your hand-labelled golden set:", golden.shape)
else:
    template = pairs_sample.sample(
        min(GOLDEN_SIZE, len(pairs_sample)), random_state=123
    )[["clean_customer_text", "clean_brand_reply"]].copy()
    template["true_intent"] = ""
    template["expected_action"] = ""
    template["label_notes"] = ""
    template.to_csv(golden_template_path, index=False)
    golden = None
    print(f"Created {golden_template_path}.")
    print("Hand-label it, save as golden_set_labeled.csv, then re-run this cell.")
    display(template.head(5))


**How to sample and label (put this note in your report):**
- Sampled uniformly at random from cleaned, deduplicated customer->AppleSupport pairs - not cherry-picked, so it reflects real class imbalance.
- For each message: read the customer text (and the brand's real historical reply, shown for context only, not as an answer key) and assign the single best-fitting `true_intent` using the taxonomy definitions above.
- `expected_action`: mark `escalate` for anything involving account security/loss of access, data loss, safety issues, legal/refund threats, or messages you personally could not confidently resolve with a templated reply; `auto` otherwise.
- Aim for roughly 150-250 examples; it's fine (expected) if class distribution is imbalanced - note that in the report.

## 5. Baseline 1 - Majority class (trivial baseline) + final test split

To keep this simple and avoid needing a second large labeled dataset, we split the golden set once: 70% "reference" (only used to compute the majority class and to fit the TF-IDF+LogReg baseline) and 30% held-out "final test" slice. Both baselines and the Gemini agent are then compared on the same held-out final test slice - this is the one number everything gets judged against.

In [ ]:
assert golden is not None and len(golden) >= 50, (
    "You need a hand-labelled golden_set_labeled.csv with at least ~50 rows before continuing. "
    "See Section 4."
)

golden = golden.dropna(subset=["true_intent", "expected_action"]).copy()
golden = golden[golden["true_intent"].isin(INTENTS)].reset_index(drop=True)
print("Valid labelled rows:", len(golden))

strat = golden["true_intent"] if golden["true_intent"].value_counts().min() >= 2 else None
ref_df, test_df = train_test_split(golden, test_size=0.30, random_state=SEED, stratify=strat)
ref_df = ref_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("Reference (fit) rows:", len(ref_df), " | Final test rows:", len(test_df))


In [ ]:
majority_intent = ref_df["true_intent"].mode()[0]
majority_pred = [majority_intent] * len(test_df)

majority_acc = accuracy_score(test_df["true_intent"], majority_pred)
majority_f1 = f1_score(test_df["true_intent"], majority_pred, average="macro", zero_division=0)

print("BASELINE 1: Majority class")
print("Majority intent:", majority_intent)
print("Accuracy:", round(majority_acc, 4), " | Macro F1:", round(majority_f1, 4))


## 6. Baseline 2 - TF-IDF + Logistic Regression (simple baseline)

Fit only on the small reference slice of the golden set (since that's the only ground truth we have without paying for large-scale LLM labeling). This is the honest "simple ML" baseline the assignment asks for.

In [ ]:
baseline_tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=3000, min_df=1)
X_ref = baseline_tfidf.fit_transform(ref_df["clean_customer_text"])
X_test = baseline_tfidf.transform(test_df["clean_customer_text"])

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_ref, ref_df["true_intent"])

logreg_pred = logreg.predict(X_test)
logreg_acc = accuracy_score(test_df["true_intent"], logreg_pred)
logreg_f1 = f1_score(test_df["true_intent"], logreg_pred, average="macro", zero_division=0)

print("BASELINE 2: TF-IDF + Logistic Regression")
print("Accuracy:", round(logreg_acc, 4), " | Macro F1:", round(logreg_f1, 4))
print()
print(classification_report(test_df["true_intent"], logreg_pred, zero_division=0))


## 7. Retrieval index - ground replies in real historical resolutions

We build a TF-IDF index over the (larger) working subsample of past customer messages, so we can retrieve similar past cases and show the model how AppleSupport actually resolved them. This index is built only from `pairs_sample` (never from the golden set) to avoid leakage.

In [ ]:
retrieval_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=8000, min_df=2)
retrieval_matrix = retrieval_vectorizer.fit_transform(pairs_sample["clean_customer_text"])

def retrieve_similar_cases(customer_text, top_k=3):
    query_vec = retrieval_vectorizer.transform([clean_text(customer_text)])
    sims = cosine_similarity(query_vec, retrieval_matrix)[0]
    top_idx = sims.argsort()[::-1][:top_k]
    cases = []
    for i in top_idx:
        cases.append({
            "customer_text": pairs_sample.iloc[i]["clean_customer_text"],
            "brand_reply": pairs_sample.iloc[i]["clean_brand_reply"],
            "similarity": float(sims[i]),
        })
    return cases

# quick sanity check
example = golden.iloc[0]["clean_customer_text"]
print("Query:", example)
for c in retrieve_similar_cases(example, top_k=2):
    print(f"  sim={c['similarity']:.3f}  ->  {c['brand_reply'][:100]}")


## 8. The agent: Gemini classifies, drafts a grounded reply, and decides auto/escalate

One structured call per message returns intent + confidence + drafted reply + escalation decision + reason, all in one shot, grounded on the retrieved similar cases.

In [ ]:
IntentLiteral = Literal[tuple(INTENTS)]

class AgentOutput(BaseModel):
    intent: IntentLiteral
    intent_confidence: float
    drafted_reply: str
    action: Literal["auto", "escalate"]
    action_reason: str

def build_prompt(customer_text, retrieved_cases):
    taxonomy = "\n".join(f"- {n}: {INTENT_DEFINITIONS[n]}" for n in INTENTS)
    cases_block = "\n\n".join(
        f"Similar past customer message: {c['customer_text']}\n"
        f"How AppleSupport actually replied: {c['brand_reply']}"
        for c in retrieved_cases
    ) or "(no similar past cases found)"

    return f"""You are an AI support agent for {BRAND} handling a customer message on Twitter.

TASK:
1. Classify the customer's PRIMARY problem into exactly one intent from the taxonomy below.
2. Draft a short, on-brand reply, grounded in how {BRAND} has actually resolved similar issues in the past (shown below). Do not invent policies or promises not supported by the examples or by publicly reasonable Apple support practice. If you are not confident a safe reply can be given, still draft your best attempt but note in the reply that a specialist may follow up.
3. Decide whether this should be handled automatically ("auto") or handed to a human ("escalate"). Escalate for: account security/access loss, data loss, safety issues, legal/refund/billing disputes, or messages you cannot confidently resolve with a templated reply. Give a concrete, specific reason (not a generic sentence).

INTENT TAXONOMY:
{taxonomy}

RETRIEVED SIMILAR PAST CASES (for grounding your reply, not ground truth to copy verbatim):
{cases_block}

CUSTOMER MESSAGE:
"{customer_text}"

Return intent_confidence as your own calibrated probability (0-1) that the intent label is correct.
"""

def run_agent(customer_text, top_k=3):
    cases = retrieve_similar_cases(customer_text, top_k=top_k)
    prompt = build_prompt(customer_text, cases)
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AgentOutput,
        ),
    )
    result = AgentOutput.model_validate_json(response.text)
    return result, cases

# quick sanity check
result, cases = run_agent(golden.iloc[0]["clean_customer_text"])
print(result.model_dump_json(indent=2))


## 9. Run the agent over the final golden test slice

This is the headline evaluation run - the held-out `test_df` slice from Section 5, never used to fit either baseline.

In [ ]:
import time

agent_rows = []
for _, row in test_df.iterrows():
    try:
        result, cases = run_agent(row["clean_customer_text"])
    except Exception as exc:
        print("Skipping a row due to error:", exc)
        continue
    agent_rows.append({
        "clean_customer_text": row["clean_customer_text"],
        "true_intent": row["true_intent"],
        "expected_action": row["expected_action"],
        "predicted_intent": result.intent,
        "intent_confidence": result.intent_confidence,
        "drafted_reply": result.drafted_reply,
        "predicted_action": result.action,
        "action_reason": result.action_reason,
        "top_similarity": max((c["similarity"] for c in cases), default=0.0),
    })
    time.sleep(0.2)  # gentle on rate limits

agent_eval = pd.DataFrame(agent_rows)
agent_eval.to_csv("agent_eval_results.csv", index=False)
print("Agent evaluated on", len(agent_eval), "golden test examples")
agent_eval.head(5)


In [ ]:
agent_intent_acc = accuracy_score(agent_eval["true_intent"], agent_eval["predicted_intent"])
agent_intent_f1 = f1_score(agent_eval["true_intent"], agent_eval["predicted_intent"], average="macro", zero_division=0)
agent_action_acc = accuracy_score(agent_eval["expected_action"], agent_eval["predicted_action"])

print("======================================")
print("HEADLINE RESULTS (held-out golden test slice, n =", len(agent_eval), ")")
print("======================================")
print(f"{'Method':35s}  Accuracy   Macro-F1")
print(f"{'Majority class (trivial)':35s}  {majority_acc:.4f}    {majority_f1:.4f}")
print(f"{'TF-IDF + LogisticRegression':35s}  {logreg_acc:.4f}    {logreg_f1:.4f}")
print(f"{'Gemini agent (this pipeline)':35s}  {agent_intent_acc:.4f}    {agent_intent_f1:.4f}")
print()
print("Auto/escalate decision accuracy vs expected_action:", round(agent_action_acc, 4))
print()
print("Full intent classification report (Gemini agent):")
print(classification_report(agent_eval["true_intent"], agent_eval["predicted_intent"], zero_division=0))


## 10. LLM-as-judge: reply quality rubric

We score each drafted reply on a 1-5 rubric (grounding, correctness, tone, actionability) using Gemini as a judge, then manually re-score a small sample yourself to check judge/human agreement - put that agreement number in your report.

In [ ]:
class JudgeScore(BaseModel):
    grounded_in_history: int  # 1-5: does it match how the brand actually resolves this kind of issue?
    factually_safe: int       # 1-5: avoids inventing unsupported promises/policies
    tone_and_clarity: int     # 1-5: on-brand, clear, empathetic
    actionable: int           # 1-5: gives the customer a concrete next step
    overall: int              # 1-5 holistic score
    justification: str

JUDGE_PROMPT = """You are grading a customer-support reply for the brand {brand}.

CUSTOMER MESSAGE:
{customer_text}

DRAFTED REPLY TO GRADE:
{drafted_reply}

REFERENCE (how a similar past case was actually resolved, for context only):
{reference}

Score the drafted reply 1 (poor) to 5 (excellent) on each dimension. Be strict: a generic
non-answer or an invented policy should score low on the relevant dimension.
"""

def judge_reply(customer_text, drafted_reply, reference):
    prompt = JUDGE_PROMPT.format(
        brand=BRAND, customer_text=customer_text, drafted_reply=drafted_reply, reference=reference
    )
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=JudgeScore),
    )
    return JudgeScore.model_validate_json(response.text)

judge_rows = []
for _, row in agent_eval.iterrows():
    cases = retrieve_similar_cases(row["clean_customer_text"], top_k=1)
    reference = cases[0]["brand_reply"] if cases else "(none found)"
    try:
        score = judge_reply(row["clean_customer_text"], row["drafted_reply"], reference)
    except Exception as exc:
        print("Judge call failed, skipping:", exc)
        continue
    judge_rows.append({**row.to_dict(), **score.model_dump()})
    time.sleep(0.2)

judged = pd.DataFrame(judge_rows)
judged.to_csv("judged_replies.csv", index=False)

print("Average LLM-judge scores across", len(judged), "replies:")
print(judged[["grounded_in_history", "factually_safe", "tone_and_clarity", "actionable", "overall"]].mean().round(2))


### Human-agreement check (do this manually)

1. Open `judged_replies.csv`.
2. Pick a random ~20-30 row sample.
3. Score `overall` yourself (1-5) blind to the judge's score, in a new column `human_overall`.
4. Come back and run the cell below to compute agreement.

Report the resulting agreement number in your report's evaluation-harness section - this is required evidence of how well your judge agrees with a human.

In [ ]:
human_scored_path = Path("judged_replies_human_scored.csv")

if human_scored_path.exists():
    human_scored = pd.read_csv(human_scored_path)
    both = human_scored.dropna(subset=["overall", "human_overall"])
    exact_agreement = (both["overall"] == both["human_overall"]).mean()
    within_one = (both["overall"] - both["human_overall"]).abs().le(1).mean()
    corr = both["overall"].corr(both["human_overall"])
    print("Rows compared:", len(both))
    print("Exact agreement:", round(exact_agreement, 3))
    print("Within +/-1 point agreement:", round(within_one, 3))
    print("Pearson correlation:", round(corr, 3))
else:
    print("Add a 'human_overall' column to a sample of judged_replies.csv, save as")
    print("judged_replies_human_scored.csv, and re-run this cell.")


## 11. Failure analysis

Real examples of intent misclassification and low-quality replies - use these directly in the report's required top-5 failure modes section.

In [ ]:
errors = agent_eval[agent_eval["true_intent"] != agent_eval["predicted_intent"]].copy()
print("Intent misclassifications:", len(errors), "out of", len(agent_eval))
display(errors[["clean_customer_text", "true_intent", "predicted_intent", "intent_confidence"]].head(20))

if len(errors) > 0:
    confusion_pairs = (
        errors.groupby(["true_intent", "predicted_intent"]).size()
        .reset_index(name="count").sort_values("count", ascending=False)
    )
    print("\nMost common confusion pairs:")
    display(confusion_pairs.head(10))

if len(judged) > 0:
    low_quality = judged[judged["overall"] <= 2].copy()
    print("\nLow LLM-judge-quality replies:", len(low_quality))
    display(low_quality[["clean_customer_text", "drafted_reply", "overall", "justification"]].head(10))


## 12. Save artifacts

In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(baseline_tfidf, MODEL_DIR / "baseline_tfidf.pkl")
joblib.dump(logreg, MODEL_DIR / "baseline_logreg.pkl")
joblib.dump(retrieval_vectorizer, MODEL_DIR / "retrieval_vectorizer.pkl")
pairs_sample.to_csv(MODEL_DIR / "retrieval_corpus.csv", index=False)

print("Saved:")
for p in sorted(MODEL_DIR.iterdir()):
    print(" -", p)


## 13. Tiny live demo (Gradio)

Type a customer message and see the agent's predicted intent, confidence, drafted reply, and auto/escalate decision - useful for a live walkthrough.

In [ ]:
import gradio as gr

def demo_fn(customer_message):
    if not customer_message.strip():
        return "", "", "", ""
    result, cases = run_agent(customer_message)
    grounding = "\n\n".join(
        f"(sim={c['similarity']:.2f}) Past reply: {c['brand_reply']}" for c in cases
    )
    return (
        result.intent,
        f"{result.intent_confidence:.2f}",
        result.drafted_reply,
        f"{result.action.upper()} - {result.action_reason}\n\nGrounding used:\n{grounding}",
    )

demo = gr.Interface(
    fn=demo_fn,
    inputs=gr.Textbox(label="Customer message", placeholder="e.g. my iphone wont update to the latest ios, keeps failing"),
    outputs=[
        gr.Textbox(label="Predicted intent"),
        gr.Textbox(label="Intent confidence"),
        gr.Textbox(label="Drafted reply", lines=4),
        gr.Textbox(label="Auto/Escalate decision + reason + grounding", lines=6),
    ],
    title=f"{BRAND} Support Agent - Live Demo",
)

demo.launch(share=False, debug=False)


## 14. Report checklist

Use this notebook's outputs directly when writing `report.md` (see the repo's report template):

- **Problem framing** - what "good" means for AppleSupport (correct intent, historically grounded reply, conservative escalation) and what you deliberately didn't build (e.g. multi-turn conversation memory, non-English support).
- **Baselines** - the headline table in Section 9: majority class, TF-IDF+LogReg, Gemini agent.
- **Headline number** - held-out golden-test-slice accuracy/macro-F1 from Section 9. State the test-slice size (it's small - roughly 60 rows out of a 200-row golden set split 70/30) as a caveat.
- **"What's misleading about my headline number?"** - the golden test slice is small (~60 rows) and drawn from one brand/time period; scores will have wide variance and may not generalize to other brands, languages, or the full 3M-row dataset; the LLM-judge is not a substitute for full human review of every reply.
- **Failure analysis** - Section 11's confusion pairs and low-quality replies, with your own hypotheses for each.
- **Next week** - e.g., expand golden set, add multi-turn context, calibrate confidence thresholds against precision/recall tradeoffs, test a second brand.
- **Decision log** - see `decision_log.md`.
